# 🎵 Idle 상태 컬럼 분류 모델 (독립 실행 버전)

이 노트북은 **완전히 독립적으로 실행** 가능합니다. 모든 의존성이 포함되어 있습니다.

- Waveform과 Mel Spectrogram을 모두 사용
- Mel Spectrogram 마스킹을 통한 중요 영역 강조
- 데이터 증강 포함
- 앙상블 모델로 예측

## 📋 목차
1. **필수 라이브러리 및 유틸리티 정의**
2. **데이터 로드**: Idle 상태 데이터만 필터링
3. **Waveform 및 Mel Spectrogram 추출**
4. **데이터 증강**
5. **마스크 생성 및 시각화**
6. **모델 정의**: Waveform CNN + Masked Mel Spectrogram CNN
7. **앙상블 모델**: Vote 방식 결합
8. **학습 및 평가**

In [1]:
# ============================================================
# 필수 라이브러리 임포트
# ============================================================

import os
import sys
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict
from typing import Optional, Tuple, List, Dict
from dataclasses import dataclass
from enum import Enum

import librosa
import librosa.display

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# 머신러닝
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 라이브러리 로드 완료!")
print(f"🖥️ Device: {device}")

✅ 라이브러리 로드 완료!
🖥️ Device: cpu


In [2]:
# ============================================================
# 유틸리티 함수 정의
# ============================================================

def get_data_dir() -> Path:
    """데이터 디렉토리 경로 반환"""
    return Path('../data')

print("✅ 유틸리티 함수 정의 완료!")

✅ 유틸리티 함수 정의 완료!


In [3]:
# ============================================================
# AudioConfig 및 AudioFeatureExtractor 클래스 정의
# ============================================================

@dataclass
class AudioConfig:
    """오디오 처리 설정"""
    sample_rate: int = 22050
    duration: float = 5.0  # 초
    n_mfcc: int = 40
    n_mels: int = 128
    n_fft: int = 2048
    hop_length: int = 512
    n_chroma: int = 12


class AudioFeatureExtractor:
    """오디오 피처 추출기"""
    
    def __init__(self, config: Optional[AudioConfig] = None):
        self.config = config or AudioConfig()
        
    def extract_mel_spectrogram(
        self, 
        y: np.ndarray, 
        sr: int,
        to_db: bool = True
    ) -> np.ndarray:
        """Mel Spectrogram 추출"""
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=self.config.n_mels,
            n_fft=self.config.n_fft,
            hop_length=self.config.hop_length
        )
        
        if to_db:
            mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
            
        return mel_spec

print("✅ AudioFeatureExtractor 클래스 정의 완료!")

✅ AudioFeatureExtractor 클래스 정의 완료!


In [4]:
# ============================================================
# AugmentationConfig 및 AudioAugmentor 클래스 정의
# ============================================================

@dataclass
class AugmentationConfig:
    """증강 설정"""
    time_stretch_rate_min: float = 0.8
    time_stretch_rate_max: float = 1.2
    pitch_shift_steps_min: int = -4
    pitch_shift_steps_max: int = 4
    noise_factor_min: float = 0.001
    noise_factor_max: float = 0.015
    volume_factor_min: float = 0.5
    volume_factor_max: float = 1.5
    time_shift_max: float = 0.2


class AudioAugmentor:
    """오디오 데이터 증강기"""
    
    def __init__(self, config: Optional[AugmentationConfig] = None):
        self.config = config or AugmentationConfig()
        
    def time_stretch(self, y: np.ndarray, rate: Optional[float] = None) -> np.ndarray:
        if rate is None:
            rate = random.uniform(self.config.time_stretch_rate_min, self.config.time_stretch_rate_max)
        if rate <= 0:
            raise ValueError(f"time_stretch rate must be positive, got {rate}")
        if abs(rate - 1.0) < 1e-6:
            return y.copy()
        return librosa.effects.time_stretch(y, rate=rate)
    
    def pitch_shift(self, y: np.ndarray, sr: int, n_steps: Optional[int] = None) -> np.ndarray:
        if n_steps is None:
            n_steps = random.randint(self.config.pitch_shift_steps_min, self.config.pitch_shift_steps_max)
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
    
    def add_noise(self, y: np.ndarray, noise_factor: Optional[float] = None) -> np.ndarray:
        if noise_factor is None:
            noise_factor = random.uniform(self.config.noise_factor_min, self.config.noise_factor_max)
        noise = np.random.randn(len(y))
        return y + noise_factor * noise
    
    def change_volume(self, y: np.ndarray, factor: Optional[float] = None) -> np.ndarray:
        if factor is None:
            factor = random.uniform(self.config.volume_factor_min, self.config.volume_factor_max)
        return y * factor
    
    def time_shift(self, y: np.ndarray, shift_max: Optional[float] = None) -> np.ndarray:
        if shift_max is None:
            shift_max = self.config.time_shift_max
        shift = int(len(y) * random.uniform(-shift_max, shift_max))
        return np.roll(y, shift)
    
    def augment(self, y: np.ndarray, sr: int, augmentations: Optional[List[str]] = None) -> np.ndarray:
        if augmentations is None:
            augmentations = ['time_stretch', 'pitch_shift', 'add_noise', 'change_volume', 'time_shift']
        
        augmented = y.copy()
        
        for aug_name in augmentations:
            if random.random() > 0.5:
                continue
            
            if aug_name == 'time_stretch':
                augmented = self.time_stretch(augmented)
            elif aug_name == 'pitch_shift':
                augmented = self.pitch_shift(augmented, sr)
            elif aug_name == 'add_noise':
                augmented = self.add_noise(augmented)
            elif aug_name == 'change_volume':
                augmented = self.change_volume(augmented)
            elif aug_name == 'time_shift':
                augmented = self.time_shift(augmented)
                
        return augmented

print("✅ AudioAugmentor 클래스 정의 완료!")

✅ AudioAugmentor 클래스 정의 완료!


---
## 1. Idle 상태 데이터 로드

In [5]:
# ============================================================
# Idle 상태 데이터만 로드 (Combined 제외)
# ============================================================

data_dir = get_data_dir()

# 오디오 파일 수집 (Idle 상태만, combined 제외)
all_files = []
all_labels = []

# 디렉토리 이름 확인 (idle state 또는 idle)
idle_dir = None
for state_dir in sorted(data_dir.iterdir()):
    if state_dir.is_dir() and 'idle' in state_dir.name.lower():
        idle_dir = state_dir
        break

if idle_dir is not None and idle_dir.exists() and idle_dir.is_dir():
    print(f"✅ Idle 상태 디렉토리 찾음: {idle_dir.name}")
    for problem_dir in sorted(idle_dir.iterdir()):
        if not problem_dir.is_dir():
            continue
        
        problem_name = problem_dir.name
        
        # Combined 클래스 제외
        if 'combined' in problem_name.lower():
            continue
        
        # 하위 폴더 확인 (combined 제외)
        has_combined = False
        try:
            for sub_dir in problem_dir.iterdir():
                if sub_dir.is_dir() and 'combined' in sub_dir.name.lower():
                    has_combined = True
                    break
        except:
            pass
        
        if has_combined:
            continue
        
        # WAV 파일 수집
        wav_files = list(problem_dir.glob('*.wav'))
        
        for wav_file in wav_files:
            all_files.append(wav_file)
            all_labels.append(problem_name)
        
        # 하위 폴더의 WAV 파일 수집 (combined 제외)
        try:
            for sub_dir in problem_dir.iterdir():
                if sub_dir.is_dir() and 'combined' not in sub_dir.name.lower():
                    sub_wav_files = list(sub_dir.glob('*.wav'))
                    for wav_file in sub_wav_files:
                        all_files.append(wav_file)
                        all_labels.append(f"{problem_name}/{sub_dir.name}")
        except:
            pass
else:
    print(f"⚠️ Idle 상태 디렉토리를 찾을 수 없습니다!")
    print(f"   확인된 디렉토리:")
    for d in sorted(data_dir.iterdir()):
        if d.is_dir():
            print(f"     - {d.name}")

print(f"\n📊 Idle 상태 데이터: {len(all_files)}개")
if len(all_files) > 0:
    print(f"📊 컬럼별 분포:")
    column_counts = Counter(all_labels)
    for column, count in sorted(column_counts.items()):
        print(f"  {column}: {count}개")
else:
    print("⚠️ 데이터가 없습니다. 디렉토리 경로를 확인해주세요.")

✅ Idle 상태 디렉토리 찾음: idle state

📊 Idle 상태 데이터: 616개
📊 컬럼별 분포:
  low_oil: 107개
  normal_engine_idle: 264개
  power_steering: 129개
  serpentine_belt: 116개


---
## 2. Waveform 및 Mel Spectrogram 추출

In [ ]:
# ============================================================
# 오디오 특징 추출 설정
# ============================================================

print("📊 실제 오디오 길이 확인 중...")
durations = []
for file_path in tqdm(all_files[:100] if len(all_files) > 100 else all_files, desc="길이 확인"):
    try:
        y, sr = librosa.load(str(file_path), sr=None)
        duration = len(y) / sr
        durations.append(duration)
    except:
        continue

if len(durations) > 0:
    durations = np.array(durations)
    print(f"\n📈 오디오 길이 통계:")
    print(f"  • 평균 길이: {np.mean(durations):.2f}초")
    print(f"  • 최소 길이: {np.min(durations):.2f}초")
    print(f"  • 최대 길이: {np.max(durations):.2f}초")
    print(f"  • 표준편차: {np.std(durations):.2f}초")
    
    max_duration = max(np.max(durations) * 1.2, 2.0)
    recommended_duration = min(max_duration, 2.0)
    print(f"\n💡 권장 duration: {recommended_duration:.1f}초")
else:
    recommended_duration = 2.0
    print(f"⚠️ 오디오 길이를 확인할 수 없어 기본값 사용: {recommended_duration}초")

audio_config = AudioConfig(
    sample_rate=22050,
    duration=recommended_duration,
    n_mels=128,
    n_fft=2048,
    hop_length=512
)

feature_extractor = AudioFeatureExtractor(audio_config)

print(f"\n🔄 Waveform 및 Mel Spectrogram 추출 중... (duration: {audio_config.duration}초)")

# Waveform 및 Mel Spectrogram 추출
X_waveform = []
X_mel = []
y_columns = []
actual_durations = []

for file_path, label in tqdm(zip(all_files, all_labels), total=len(all_files)):
    try:
        y, sr = librosa.load(str(file_path), sr=audio_config.sample_rate)
        
        actual_duration = len(y) / sr
        actual_durations.append(actual_duration)
        
        target_length = int(audio_config.sample_rate * audio_config.duration)
        if len(y) < target_length:
            y_padded = np.pad(y, (0, target_length - len(y)), mode='constant')
        elif len(y) > target_length:
            y_padded = y[:target_length]
        else:
            y_padded = y
        
        X_waveform.append(y_padded)
        
        mel_spec = feature_extractor.extract_mel_spectrogram(y, sr, to_db=False)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        X_mel.append(mel_spec_db)
        
        y_columns.append(label)
        
    except Exception as e:
        print(f"⚠️ 오류 발생: {file_path} - {e}")
        continue

X_waveform = np.array(X_waveform)
X_mel = np.array(X_mel)
y_columns = np.array(y_columns)
actual_durations = np.array(actual_durations)

print(f"\n✅ 추출 완료!")
print(f"  Waveform shape: {X_waveform.shape}")
print(f"  Mel Spectrogram shape: {X_mel.shape}")
print(f"  컬럼 레이블 shape: {y_columns.shape}")
print(f"  실제 오디오 평균 길이: {np.mean(actual_durations):.2f}초")

# 원본 데이터 저장 (마스크 생성용)
X_waveform_original = X_waveform.copy()
X_mel_original = X_mel.copy()
y_columns_original = y_columns.copy()
print(f"  📌 원본 데이터 저장 완료 (마스크 생성용)")

📊 실제 오디오 길이 확인 중...


길이 확인:   0%|          | 0/100 [00:00<?, ?it/s]

---
## 3. 데이터 증강

In [ ]:
# ============================================================
# 데이터 증강 (Data Augmentation)
# ============================================================

print("\n🔄 데이터 증강 시작...")

unique_columns = sorted(np.unique(y_columns))
column_counts = {}
for column in unique_columns:
    column_counts[column] = np.sum(y_columns == column)

print("\n📊 증강 전 컬럼별 개수:")
for column, count in column_counts.items():
    print(f"  {column}: {count}개")

max_count = max(column_counts.values())
target_count = int(max_count * 1.2)
print(f"\n🎯 목표 개수: 각 클래스당 {target_count}개")

augmentation_config = AugmentationConfig(
    time_stretch_rate_min=0.85,
    time_stretch_rate_max=1.15,
    pitch_shift_steps_min=-3,
    pitch_shift_steps_max=3,
    noise_factor_min=0.002,
    noise_factor_max=0.01,
    volume_factor_min=0.7,
    volume_factor_max=1.3,
    time_shift_max=0.15
)
augmentor = AudioAugmentor(config=augmentation_config)

augmented_waveforms = []
augmented_mels = []
augmented_labels = []

for column in unique_columns:
    column_indices = np.where(y_columns == column)[0]
    current_count = len(column_indices)
    needed_count = target_count - current_count
    
    print(f"\n  {column}: {current_count}개 → {target_count}개 (증강 필요: {needed_count}개)")
    
    if needed_count <= 0:
        continue
    
    augmented_count = 0
    sample_idx = 0
    
    file_list_for_column = [(i, f) for i, (f, lbl) in enumerate(zip(all_files, all_labels)) if lbl == column]
    
    while augmented_count < needed_count:
        original_data_idx = column_indices[sample_idx % current_count]
        sample_idx += 1
        
        try:
            data_order_in_column = np.where(column_indices == original_data_idx)[0][0]
            
            if data_order_in_column < len(file_list_for_column):
                original_file = file_list_for_column[data_order_in_column][1]
            else:
                original_file = file_list_for_column[data_order_in_column % len(file_list_for_column)][1]
            
            y_original, sr = librosa.load(str(original_file), sr=audio_config.sample_rate)
            y_augmented = augmentor.augment(y_original, sr)
            
            target_length = int(audio_config.sample_rate * audio_config.duration)
            if len(y_augmented) < target_length:
                y_augmented_padded = np.pad(y_augmented, (0, target_length - len(y_augmented)), mode='constant')
            elif len(y_augmented) > target_length:
                y_augmented_padded = y_augmented[:target_length]
            else:
                y_augmented_padded = y_augmented
            
            augmented_waveforms.append(y_augmented_padded)
            
            mel_spec_aug = feature_extractor.extract_mel_spectrogram(y_augmented, sr, to_db=False)
            mel_spec_aug_db = librosa.power_to_db(mel_spec_aug, ref=np.max)
            
            if len(X_mel) > 0:
                target_time_frames = X_mel[0].shape[1]
                current_time_frames = mel_spec_aug_db.shape[1]
                
                if current_time_frames < target_time_frames:
                    padding = np.zeros((mel_spec_aug_db.shape[0], target_time_frames - current_time_frames))
                    padding.fill(mel_spec_aug_db.min())
                    mel_spec_aug_db = np.concatenate([mel_spec_aug_db, padding], axis=1)
                elif current_time_frames > target_time_frames:
                    mel_spec_aug_db = mel_spec_aug_db[:, :target_time_frames]
            
            augmented_mels.append(mel_spec_aug_db)
            augmented_labels.append(column)
            augmented_count += 1
            
        except Exception as e:
            print(f"    ⚠️ 증강 오류: {e}")
            continue

if len(augmented_waveforms) > 0:
    X_waveform = np.concatenate([X_waveform, np.array(augmented_waveforms)], axis=0)
    X_mel = np.concatenate([X_mel, np.array(augmented_mels)], axis=0)
    y_columns = np.concatenate([y_columns, np.array(augmented_labels)], axis=0)
    
    print(f"\n✅ 데이터 증강 완료!")
    print(f"  증강된 샘플: {len(augmented_waveforms)}개")
    print(f"  총 샘플: {len(X_waveform)}개")
    
    print(f"\n📊 증강 후 컬럼별 개수:")
    for column in unique_columns:
        count = np.sum(y_columns == column)
        print(f"  {column}: {count}개")
else:
    print("\n⚠️ 증강된 데이터가 없습니다.")

---
## 4. 마스크 생성 및 시각화

In [ ]:
# ============================================================
# 컬럼별 평균 Mel Spectrogram 계산 (원본 데이터만 사용)
# ============================================================

unique_columns_original = sorted(np.unique(y_columns_original))

column_means = {}
for column in unique_columns_original:
    column_indices = np.where(y_columns_original == column)[0]
    if len(column_indices) > 0:
        column_means[column] = np.mean(X_mel_original[column_indices], axis=0)

print(f"✅ 컬럼별 평균 Mel Spectrogram 계산 완료! ({len(column_means)}개 컬럼)")
print(f"  📌 원본 데이터만 사용하여 마스크 생성")

In [ ]:
# ============================================================
# 중요 영역 마스크 생성 (원본 데이터 기반)
# ============================================================

importance_mask = np.zeros_like(X_mel_original[0])

columns_list = list(column_means.keys())
for i in range(len(columns_list)):
    for j in range(i + 1, len(columns_list)):
        diff = np.abs(column_means[columns_list[i]] - column_means[columns_list[j]])
        importance_mask = np.maximum(importance_mask, diff)

if importance_mask.max() > importance_mask.min():
    importance_mask = (importance_mask - importance_mask.min()) / (importance_mask.max() - importance_mask.min())
else:
    importance_mask = np.ones_like(importance_mask) * 0.5

importance_mask_tensor = torch.FloatTensor(importance_mask).unsqueeze(0).to(device)

print(f"✅ 중요 영역 마스크 생성 완료!")
print(f"  마스크 shape: {importance_mask.shape}")
print(f"  마스크 범위: [{importance_mask.min():.3f}, {importance_mask.max():.3f}]")

---
## 5. 모델 정의

In [ ]:
# ============================================================
# Waveform 1D CNN 모델
# ============================================================

class WaveformCNN1D(nn.Module):
    def __init__(self, num_classes: int, input_length: int = 110250, base_channels: int = 64, dropout: float = 0.3):
        super().__init__()
        self.num_classes = num_classes
        
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(dropout / 2)
        )
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Sequential(
            nn.Linear(base_channels * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc_out = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        return x

print("✅ WaveformCNN1D 모델 정의 완료!")

In [ ]:
# ============================================================
# 마스킹 기반 Mel Spectrogram CNN 모델
# ============================================================

class MaskedSpatialAttention(nn.Module):
    def __init__(self, importance_mask: torch.Tensor, learnable: bool = True):
        super().__init__()
        self.importance_mask = nn.Parameter(importance_mask, requires_grad=learnable)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mask = self.importance_mask.expand(x.size(0), -1, -1)
        mask = mask.unsqueeze(1)
        x = x * (1 + mask)
        return x


class MaskedCNN(nn.Module):
    def __init__(self, num_classes: int, importance_mask: torch.Tensor, in_channels: int = 1, base_channels: int = 32, dropout: float = 0.3):
        super().__init__()
        self.num_classes = num_classes
        self.masked_attention = MaskedSpatialAttention(importance_mask, learnable=True)
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, padding=1),
            nn.BatchNorm2d(base_channels * 8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout)
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.fc1 = nn.Sequential(
            nn.Linear(base_channels * 8, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        
        self.fc_out = nn.Linear(128, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.masked_attention(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        return x

print("✅ MaskedCNN 모델 정의 완료!")

In [ ]:
# ============================================================
# 앙상블 모델 클래스 (Vote 방식)
# ============================================================

class EnsembleVoteModel(nn.Module):
    def __init__(self, waveform_model: nn.Module, spectrogram_model: nn.Module, vote_method: str = 'soft'):
        super().__init__()
        self.waveform_model = waveform_model
        self.spectrogram_model = spectrogram_model
        self.vote_method = vote_method
    
    def forward(self, waveform_input, spectrogram_input):
        waveform_output = self.waveform_model(waveform_input)
        spec_output = self.spectrogram_model(spectrogram_input)
        
        if self.vote_method == 'hard':
            waveform_pred = waveform_output.argmax(dim=1)
            spec_pred = spec_output.argmax(dim=1)
            ensemble_pred = torch.stack([waveform_pred, spec_pred], dim=1)
            ensemble_pred = torch.mode(ensemble_pred, dim=1)[0]
            ensemble_output = F.one_hot(ensemble_pred, num_classes=waveform_output.size(1)).float()
            ensemble_output = ensemble_output * 10.0 - 5.0
        else:
            ensemble_output = (waveform_output + spec_output) / 2
        
        return ensemble_output

print("✅ EnsembleVoteModel 클래스 정의 완료!")

In [ ]:
# ============================================================
# 데이터셋 클래스
# ============================================================

class ColumnClassificationDataset(Dataset):
    def __init__(self, X_waveform, X_mel, y_columns, column_to_idx):
        self.X_waveform = X_waveform
        self.X_mel = X_mel
        self.y_columns = y_columns
        self.column_to_idx = column_to_idx
        self.y_indices = np.array([column_to_idx[column] for column in y_columns])
    
    def __len__(self):
        return len(self.X_waveform)
    
    def __getitem__(self, idx):
        waveform = torch.FloatTensor(self.X_waveform[idx]).unsqueeze(0)
        mel_spec = torch.FloatTensor(self.X_mel[idx]).unsqueeze(0)
        label = torch.LongTensor([self.y_indices[idx]])[0]
        return waveform, mel_spec, label

print("✅ ColumnClassificationDataset 클래스 정의 완료!")

---
## 6. 데이터 분할 및 모델 학습

In [ ]:
# ============================================================
# 컬럼 레이블 매핑
# ============================================================

unique_columns = sorted(np.unique(y_columns))
column_to_idx = {column: idx for idx, column in enumerate(unique_columns)}
idx_to_column = {idx: column for column, idx in column_to_idx.items()}
num_classes = len(unique_columns)

print(f"📊 컬럼 클래스:")
for column, idx in column_to_idx.items():
    count = np.sum(y_columns == column)
    print(f"  {column}: {count}개 (인덱스: {idx})")

print(f"\n총 클래스 수: {num_classes}개")

In [ ]:
# ============================================================
# Train/Val/Test 분할
# ============================================================

X_waveform_train, X_waveform_test, X_mel_train, X_mel_test, y_train, y_test = train_test_split(
    X_waveform, X_mel, y_columns, test_size=0.2, random_state=42, stratify=y_columns
)

X_waveform_train, X_waveform_val, X_mel_train, X_mel_val, y_train, y_val = train_test_split(
    X_waveform_train, X_mel_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"📊 데이터 분할:")
print(f"  Train: {len(X_waveform_train)}개")
print(f"  Val: {len(X_waveform_val)}개")
print(f"  Test: {len(X_waveform_test)}개")

In [ ]:
# ============================================================
# 데이터셋 및 DataLoader 생성
# ============================================================

train_dataset = ColumnClassificationDataset(X_waveform_train, X_mel_train, y_train, column_to_idx)
val_dataset = ColumnClassificationDataset(X_waveform_val, X_mel_val, y_val, column_to_idx)
test_dataset = ColumnClassificationDataset(X_waveform_test, X_mel_test, y_test, column_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("✅ 데이터셋 및 DataLoader 생성 완료!")

In [ ]:
# ============================================================
# 모델 생성
# ============================================================

waveform_input_length = X_waveform.shape[1]
waveform_model = WaveformCNN1D(
    num_classes=num_classes,
    input_length=waveform_input_length,
    base_channels=64,
    dropout=0.3
).to(device)

mel_model = MaskedCNN(
    num_classes=num_classes,
    importance_mask=importance_mask_tensor,
    in_channels=1,
    base_channels=32,
    dropout=0.3
).to(device)

ensemble_model = EnsembleVoteModel(
    waveform_model=waveform_model,
    spectrogram_model=mel_model,
    vote_method='soft'
).to(device)

print("✅ 모델 생성 완료!")
print(f"  Waveform 모델 파라미터: {sum(p.numel() for p in waveform_model.parameters()):,}개")
print(f"  Mel Spectrogram 모델 파라미터: {sum(p.numel() for p in mel_model.parameters()):,}개")
print(f"  앙상블 모델 파라미터: {sum(p.numel() for p in ensemble_model.parameters()):,}개")

In [ ]:
# ============================================================
# 학습 함수
# ============================================================

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for waveform, mel_spec, labels in train_loader:
        waveform = waveform.to(device)
        mel_spec = mel_spec.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(waveform, mel_spec)
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    
    return total_loss / len(train_loader), correct / total


def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for waveform, mel_spec, labels in val_loader:
            waveform = waveform.to(device)
            mel_spec = mel_spec.to(device)
            labels = labels.to(device)
            
            outputs = model(waveform, mel_spec)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)
            
            total_loss += loss.item()
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(val_loader), correct / total, all_preds, all_labels

print("✅ 학습/검증 함수 정의 완료!")

In [ ]:
# ============================================================
# 모델 학습
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(ensemble_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

num_epochs = 50
best_val_acc = 0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("🚀 앙상블 모델 학습 시작\n")

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(ensemble_model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = validate(ensemble_model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model_path = Path('checkpoints')
        model_path.mkdir(exist_ok=True)
        torch.save(ensemble_model.state_dict(), model_path / 'best_idle_column_model.pth')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print()

print("✅ 학습 완료!")

In [ ]:
# ============================================================
# 학습 곡선 시각화
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_accs, label='Train Acc', linewidth=2)
axes[1].plot(val_accs, label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ 학습 곡선 시각화 완료!")

In [ ]:
# ============================================================
# 최종 테스트 평가
# ============================================================

model_path = Path('checkpoints') / 'best_idle_column_model.pth'
if model_path.exists():
    ensemble_model.load_state_dict(torch.load(model_path))
    print(f"✅ 모델 로드 완료: {model_path}")
else:
    print(f"⚠️ 모델 파일을 찾을 수 없습니다: {model_path}")

test_loss, test_acc, test_preds, test_labels = validate(ensemble_model, test_loader, criterion, device)

print(f"\n📊 최종 테스트 결과:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")

print(f"\n📋 분류 리포트:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=[idx_to_column[i] for i in range(num_classes)]
))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[idx_to_column[i] for i in range(num_classes)],
            yticklabels=[idx_to_column[i] for i in range(num_classes)])
plt.title('Confusion Matrix - Idle Column Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()